# **1.1 · Load and denoise the data**

In [2]:
from pathlib import Path
import html
import re
import unicodedata
import pandas as pd

paths = [folder / name for folder in (Path.cwd(), Path('upload'))
         for name in ('research_abstracts.json', 'research_abstracts (1).json')]
path = next(p for p in paths if p.is_file())
raw = pd.read_json(path)
print('Input shape:', raw.shape)
print('Missing values:', raw.isna().sum().to_dict())
print('Original labels:', raw['Label'].value_counts().to_dict())

Input shape: (1138, 3)
Missing values: {'DOI': 4, 'Abstract': 0, 'Label': 0}
Original labels: {'classical': 900, 'ai': 238}


In [3]:
def clean_text(value):
    if pd.isna(value):
        return ''
    text = unicodedata.normalize('NFKC', html.unescape(str(value))).casefold()
    text = re.sub(r'</?[a-z][^>]*>', ' ', text)
    text = re.sub(r'(?:https?://|www\.)\S+|\b10\.\d{4,9}/\S+', ' ', text)
    text = re.sub(r'©[^\n]*$', ' ', text.strip())
    return ' '.join(re.findall(r'[^\W_]+', text))

papers = raw.copy()
papers['text'] = papers['Abstract'].map(clean_text)
papers['Label'] = papers['Label'].astype('string').str.strip().str.lower()
papers = papers.loc[papers['text'].ne('') & papers['Label'].notna() & papers['Label'].ne('')]
# Ambiguous identical abstracts cannot supply a unique training target.
consistent = papers.groupby('text')['Label'].transform('nunique').eq(1)
print('Rows with conflicting labels:', (~consistent).sum())
papers = papers.loc[consistent].drop_duplicates('text').reset_index(drop=True)
print('Retained abstracts:', len(papers))
print('Retained labels:', papers['Label'].value_counts().to_dict())
print('Example:', papers.loc[0, 'text'][:180])

Rows with conflicting labels: 0
Retained abstracts: 1121
Retained labels: {'classical': 886, 'ai': 235}
Example: forward looking forecasting of the inflation rate could help the central bank and other government departments to better use monetary policy to stabilize prices and prevent the imp


# **1.2 · Select columns and model the features**

In [4]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    papers['text'], papers['Label'].astype(str), test_size=0.20,
    stratify=papers['Label'], random_state=42)
model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
model.fit(X_train, y_train)
vectorizer, classifier = model.steps[0][1], model.steps[1][1]
terms = vectorizer.get_feature_names_out()
ranking = classifier.coef_[0].argsort()
print('Training matrix:', vectorizer.transform(X_train).shape)
for label, indices in zip(classifier.classes_, [ranking[:6], ranking[::-1][:6]]):
    print(label, 'associated features:', ', '.join(terms[indices]))

Training matrix: (896, 21571)
ai associated features: learning, neural, machine, machine learning, network, neural network
classical associated features: policy, monetary, the model, the predictions, effect, interest


# **1.3 · Evaluate learning and generalization**

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(model, X_train, y_train, cv=cv, scoring=['accuracy', 'f1_macro'])
dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
pred = model.predict(X_test)
for name, truth, guess in [
        ('Training', y_train, model.predict(X_train)),
        ('Test', y_test, pred), ('Majority rule on test', y_test, dummy.predict(X_test))]:
    print(f'{name:22s} n={len(truth):4d}  accuracy={accuracy_score(truth, guess):.3f}'
          f'  macro-F1={f1_score(truth, guess, average="macro"):.3f}')
for metric in ['accuracy', 'f1_macro']:
    values = scores['test_' + metric]
    print(f'CV {metric}: {values.mean():.3f} (fold SD {values.std():.3f})')
print(classification_report(y_test, pred, digits=3))
labels = ['ai', 'classical']
print(pd.DataFrame(confusion_matrix(y_test, pred, labels=labels),
                   index=['Actual ' + x for x in labels],
                   columns=['Predicted ' + x for x in labels]))

Training               n= 896  accuracy=0.993  macro-F1=0.990
Test                   n= 225  accuracy=0.907  macro-F1=0.862
Majority rule on test  n= 225  accuracy=0.791  macro-F1=0.442
CV accuracy: 0.886 (fold SD 0.014)
CV f1_macro: 0.820 (fold SD 0.025)
              precision    recall  f1-score   support

          ai      0.760     0.809     0.784        47
   classical      0.949     0.933     0.941       178

    accuracy                          0.907       225
   macro avg      0.854     0.871     0.862       225
weighted avg      0.909     0.907     0.908       225

                  Predicted ai  Predicted classical
Actual ai                   38                    9
Actual classical            12                  166
